# Modeling crime in USA


**"Science may be described as the art of systematic over-simplification — the art of discerning what we may with advantage omit."** (Karl Popper, The Open Universe : An Argument for Indeterminism (1992), p. 44)

In [ ]:
#sets the notebook width
from IPython.display import display, HTML


display(HTML("<style>.container { width:75% !important; }</style>"))
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings

warnings.filterwarnings('ignore')

## Packages

In [ ]:
# Standard Libraries
import sys
import os
import glob
import math
from pathlib import Path

# Make imports and relative data paths deterministic whether Jupyter was
# launched from the repository root or from notebooks/.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
(PROJECT_ROOT / 'figures' / 'sm').mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import palettable

In [ ]:
# conda install pandas scipy statsmodels scikit-learn

In [ ]:
# Third party libraries
import numpy as np
import pandas as pd
from scipy import stats
from scipy import optimize
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.metrics import r2_score
from scipy import optimize
import seaborn as sns
#import arviz as az
#import pystan

In [ ]:
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {'text.usetex': usetex,
              'font.family': 'sans-serif',
              'font.sans-serif': ['Times'],
#               'text.latex.unicode': True,
#               'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
#                                       r"\usepackage{lmodern}",
#                                       r"\usepackage{amsmath}",
#                                       r"\usepackage{mathptmx}"
#                                       ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction":
                "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=False)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

In [ ]:
# conda install -c conda-forge watermark

## Local modules 

In [ ]:
import pandas as pd 
import numpy as np

ticksnamemap={'Intercept':'Intercept',
              'Blacks':'B&AA pop.',
              'Minorities':'Other pop.',
              'Whites':'White pop.',
              'Poverty':'Pop. in poverty',
              'Murder':'Murders',
              'Population':'Population',
              'Education':'Pop. w/o JH diploma',
              'Unemployment':'Pop. unemployed',

             }

urban_metrics=['Population', 'Whites', 'Blacks', 'Minorities', 'Poverty', 'Education',
               'Unemployment', 'Murder']

def load_data():
    data = pd.read_csv('../data/clean_dataframes/full_merged_data_1960_2020.csv',
               index_col=0)
    data = data[data.Year <= 2020]
    data = data[data.Year >= 1960]

    return data

def replace_nan_by_zero(data):
    data=data[data['County'].isin(np.unique(data[((data.Year==1960) )].County.values))]
    data = data[['County','State','Year','Whites', 'Blacks', 'Minorities',
              'Poverty','Unemployment','Education', 'Murder','Population']]
    data = data.dropna()
    data=data.reset_index(drop=True)
    for metric in urban_metrics:
        # data[metric]= data[metric].replace(np.nan, 0)
        data[metric]=data[metric].astype(int) 
    return data

def to_msa(data):
    crosswalk=pd.read_csv('../data/cbsatocountycrosswalk.csv')
    crosswalk['state']=crosswalk['state']
    crosswalk=crosswalk[['countyname','state','msaname']]
    crosswalk['name']=crosswalk['countyname'].str.title()+', '+crosswalk['state']
    crosswalk['msaname']=crosswalk['msaname'].str.split(',').str[0].str.title()+', '+crosswalk['state']
    crosswalk['countyname']=crosswalk['name'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.strip()
    crosswalk['msaname']=crosswalk['msaname'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.strip()
    crosswalk=crosswalk[['countyname','msaname']]
    msamap=dict()
    for i in range(0,len(crosswalk)):
        msamap[crosswalk.loc[i].countyname]=crosswalk.loc[i].msaname

    data_msa=data.copy()
    data_msa['County']=data_msa['County'].str.normalize('NFKD').str.encode('ascii', 
                                                                             errors='ignore').str.decode('utf-8').str.strip()
    data_msa['MSA']=data_msa['County'].map(msamap)
    data_msa.dropna(inplace=True)
    data_msa.reset_index(drop=True,inplace=True)
    data_msa=data_msa.groupby(['MSA','State','Year']).sum().reset_index()
    data=data_msa
    
    states_names=pd.read_csv('../data/state_abbreviations.csv')
    states_names['full_name']=states_names['Name']+', '+states_names['Abbreviation']
    data=data[~data.MSA.isin(states_names['full_name'])]

    return data
    
def data_log_transform(data,dropna=True):
    df=data.copy()

    for metric in ['Whites', 'Blacks', 'Minorities','Poverty','Unemployment','Education', 'Murder','Population','Non-White']:
        df[metric]=(df[metric]).apply(np.log10).replace(-np.inf,0)
    df['Year']=data['Year']
    df['MSA']=data['MSA']
    df['State']=data['State']

    if dropna:
        df = df.dropna().reset_index(drop=True)
    df.to_csv('../data/log_df.csv')
    return df

# Data

In [ ]:
data=load_data()
data = data[~np.isnan(data['FIPS'])]
data['FIPS'] = data['FIPS'].astype(int).apply(lambda x: str(x).zfill(5))
data.head()

In [ ]:
data[data['State'].str.contains('NY')]

In [ ]:
data[data['County'].str.contains('New York')]

In [ ]:
data_nan = replace_nan_by_zero(data)
data_nan.head()

## Metropolitan Statistical Areas 

In [ ]:
data=to_msa(data)

In [ ]:
data

## Figure 1: Temporal evolution of crime

In [ ]:
hist_data = pd.read_csv('../data/homicide_trend.dat', sep='\t')
hist_data_csv = pd.read_csv('../data/historical_trend/homicide_rate_cdc_wonder.tsv',
                            sep='\t', header=None, names=['year', 'homicides', 'population', 'rate'])
hist_data_csv.head()

In [ ]:
aa_year=np.array([1910,1920,1930,1940,1950,1960,1970,1980,1990])
aa_pop_south=np.array([
                    8749427,
                    8912231,
                    9361577,
                    9904619,
                    10225407,
                    11311607,
                    11969961,
                    14047787,
                    15828888])

aa_pop_us=np.array([
                   9827763,
                   10463131,
                   11891143,
                   12865518,
                   15042286,
                   18860117,
                   22580289,
                   26495025,
                   29986060])

or_year=np.array([1950,1960,1970,1980,1990])

total_pop=np.array([150697361,
                    178464236,
                    203211926,
                    226545805,
                    248709873])

or_pop_us=np.array([
                   48604,
                   75045,
                   516686,
                   6758319,
                   9804847])

In [ ]:
df_year=data[['Murder','Population','Year']]
df_year=df_year.groupby(['Year']).sum().reset_index()

In [ ]:
f, ax = plt.subplots(nrows=1, ncols=1, figsize=stdfigsize(nx=2, ny=0.8, scale=1))


df=hist_data[hist_data.Year>1920]
df=df[df.Year<1951]
ax.plot(df['Year'],df['homicide rate'],'--',color='#e41a1c',alpha=1,label='NVSS (homicide) ')

df=hist_data[hist_data.Year>1950]
df=df[df.Year<2005]
ax.plot(df['Year'],df['homicide rate'],'-',color='#e41a1c',alpha=1,label='BJS (homicide)')

ax.plot(hist_data_csv['year'],hist_data_csv['rate'],'-',color='k',alpha=1,label='WONDER (homicide)')

ax.plot(df_year['Year'],df_year['Murder']/df_year['Population']*100000,'--',color='#1f78b4',alpha=1,label='FBI (murder)')

ax.set_ylabel(r"Rate per 100K")
ax.set_xlabel(r"Year, $t$")
ax.set_xlim(1920,2021)
ax.legend(labelspacing=0.05,frameon=True,loc='upper right',bbox_to_anchor=(1, 1.1))
x_ticks = np.arange(1920, 2021, 20)
ax.set_xticks(x_ticks)
y_ticks = np.arange(-3, 12, 2)
ax.set_yticks(y_ticks)
ax.grid(axis='y',linewidth=1)
    

    
ax.plot([1920, 1933],[3,3],linewidth=32,color='#feb24c',solid_capstyle='butt')
ax.plot([1941, 1945],[3,3],linewidth=32,color='lightgray',alpha=0.6,solid_capstyle='butt')
colors=['#dadaeb','#dadaeb','#3f007d','#9e9ac8']
n=len(or_pop_us/total_pop)
x=[1950,1960,1970,1980,1990,2000]
y=[2.5 for i in range(0,n)]
s = 1 # Segment length
for i in range(0,n-s,s):
    print(x[i],x[i+1])
    ax.plot([x[i],x[i+1]],[3,3],color=colors[i],alpha=1,linewidth=32,solid_capstyle='butt')
colors=['w','#74a9cf','#a6bddb','#023858','#0570b0','#3690c0']
n=len(aa_pop_south/aa_pop_us)-1
x=np.arange(1910, 1981,10)
s = 1 # Segment length
for i in range(0,n-s,s):
    if i <6:
        ax.plot([x[i],x[i+1]],[1.,1.],color=colors[i],alpha=1,linewidth=32,solid_capstyle='butt')

ax.set_ylim(0,11)

ax.text(1922., 2.8, "Prohibition",fontsize=20,color='k')
ax.text(1941, 2.8, "WWII",fontsize=20,color='k')   
ax.text(1953, 2.8, "Other ethnicities",fontsize=20,color='k')
ax.text(1923.5, 0.8, "Great migration",fontsize=20,color='k')


ax.annotate("A", xy=(-0.1, 1.05), xycoords="axes fraction",fontsize=40)
f.tight_layout()
f.savefig('../figures/fig1A.pdf',bbox_inches='tight')

In [ ]:
f, ax = plt.subplots(nrows=1, ncols=1, figsize=stdfigsize(nx=2, ny=0.8, scale=1))



cities=['New York-Newark, NY','Atlanta, GA','Los Angeles-Long Beach, CA','Chicago, IL']

ax.annotate("B", xy=(-0.1, 1.05), xycoords="axes fraction",fontsize=40)

colors=['#41ab5d','#fec44f','#3f007d','#1d91c0']
ls=['solid',(0, (1, 0.5)),(0, (5,0.8)),(0, (3,1))]
i=0
for city in cities:
    x=data[data.MSA==city].Year
    m=data[data.MSA==city].Murder
    p=data[data.MSA==city].Population
    ax.plot(x,m/p*100000,color=colors[i],label=city,linestyle=ls[i])
    i=i+1
ax.set_ylabel(r"Rate per 100K")

x_ticks = np.arange(1920, 2021, 20)
ax.set_xticks(x_ticks)
ax.set_xlabel(r"Year, $t$")

y_ticks = np.arange(5, 30, 5)
ax.set_yticks(y_ticks)
ax.grid(axis='y',linewidth=1)
ax.legend(loc='upper left',frameon=True,fontsize=25)
f.tight_layout()
# Superseded by the two-row Figure 1B layout saved below.

In [ ]:
# Create subplots with two panels (2 rows, 1 column)
f, axes = plt.subplots(nrows=2, ncols=1, figsize=stdfigsize(nx=2, ny=1.1, scale=1), sharex=True)

# Panel 1: New York and Los Angeles
ax1 = axes[0]
cities_panel1 = ['New York-Newark, NY', 'Los Angeles-Long Beach, CA']
colors_panel1 = ['#41ab5d', '#3f007d']
ls_panel1 = ['solid', (0, (5, 0.8))]

ax1.annotate("B", xy=(-0.1, 1.05), xycoords="axes fraction", fontsize=40)

i = 0
for city in cities_panel1:
    x = data[data.MSA == city].Year
    m = data[data.MSA == city].Murder
    p = data[data.MSA == city].Population
    ax1.plot(x, m/p*100000, color=colors_panel1[i], label=city, linestyle=ls_panel1[i])
    i += 1

ax1.set_xticks(np.arange(1920, 2021, 20))
ax1.set_yticks(np.arange(5, 30, 5))
ax1.grid(axis='y', linewidth=1)
ax1.legend(loc='upper left', frameon=True, fontsize=25)

# Panel 2: Atlanta and Chicago
ax2 = axes[1]
cities_panel2 = ['Atlanta, GA', 'Chicago, IL']
colors_panel2 = ['#fec44f', '#1d91c0']
ls_panel2 = [(0, (1, 0.5)), (0, (3, 1))]

i = 0
for city in cities_panel2:
    x = data[data.MSA == city].Year
    m = data[data.MSA == city].Murder
    p = data[data.MSA == city].Population
    ax2.plot(x, m/p*100000, color=colors_panel2[i], label=city, linestyle=ls_panel2[i])
    i += 1

ax2.set_xticks(np.arange(1920, 2021, 20))
ax2.set_xlabel(r"Year, $t$")
ax2.set_yticks(np.arange(5, 30, 5))
ax2.grid(axis='y', linewidth=1)
ax2.legend(loc='upper left', frameon=True, fontsize=25)

# Super ylabel for both panels
f.text(0.02, 0.6, 'Rate per 100K', va='center', rotation='vertical', fontsize=30)

# Adjust layout
f.tight_layout(pad=2.0)

# Show the plot
f.tight_layout()
#f.savefig('../figures/fig1B.pdf',bbox_inches='tight')

In [ ]:
# Create subplots with two panels (2 rows, 1 column)
f, axes = plt.subplots(nrows=2, ncols=1, figsize=stdfigsize(nx=2, ny=1.1, scale=1), sharex=True)

# Panel 1: New York and Los Angeles
ax1 = axes[0]
cities_panel1 = ['New York-Newark, NY', 'Los Angeles-Long Beach, CA']
colors_panel1 = ['#41ab5d', '#3f007d']
ls_panel1 = ['solid', (0, (5, 0.8))]

ax1.annotate("B", xy=(-0.1, 1.05), xycoords="axes fraction", fontsize=40)

i = 0
for city in cities_panel1:
    x = data[data.MSA == city].Year
    m = data[data.MSA == city].Murder
    p = data[data.MSA == city].Population
    ax1.plot(x, m/p*100000, color=colors_panel1[i], label=city, linestyle=ls_panel1[i])
    i += 1

ax1.set_xticks(np.arange(1920, 2021, 20))
ax1.set_yticks(np.arange(5, 30, 5))
ax1.grid(axis='y', linewidth=1)
ax1.annotate(cities_panel1[0],
                    xy=(0.3, 0.75), 
                    xycoords="axes fraction",
                    fontsize=30,
                    color=colors_panel1[0],zorder=-1)
ax1.annotate(cities_panel1[1],
                    xy=(0.72, 0.75), 
                    xycoords="axes fraction",
                    fontsize=30,
                    color=colors_panel1[1],zorder=-1)
#ax1.legend(loc='upper left', frameon=True, fontsize=25)

# Panel 2: Atlanta and Chicago
ax2 = axes[1]
cities_panel2 = ['Atlanta, GA', 'Chicago, IL']
colors_panel2 = ['#fec44f', '#1d91c0']
ls_panel2 = [(0, (1, 0.5)), (0, (3, 1))]

i = 0
for city in cities_panel2:
    x = data[data.MSA == city].Year
    m = data[data.MSA == city].Murder
    p = data[data.MSA == city].Population
    ax2.plot(x, m/p*100000, color=colors_panel2[i], label=city, linestyle=ls_panel2[i])
    i += 1

ax2.set_xticks(np.arange(1920, 2021, 20))
ax2.set_xlabel(r"Year, $t$")
ax2.set_yticks(np.arange(5, 30, 5))
ax2.grid(axis='y', linewidth=1)
#ax2.legend(loc='upper left', frameon=True, fontsize=25)

ax2.annotate(cities_panel2[0],
                    xy=(0.32, 0.66), 
                    xycoords="axes fraction",
                    fontsize=30,
                    color=colors_panel2[0],zorder=-1)
ax2.annotate(cities_panel2[1],
                    xy=(0.72, 0.65), 
                    xycoords="axes fraction",
                    fontsize=30,
                    color=colors_panel2[1],zorder=-1)

# Super ylabel for both panels
f.text(0.02, 0.6, 'Rate per 100K', va='center', rotation='vertical', fontsize=30)

# Adjust layout
f.tight_layout(pad=2.0)

# Show the plot
f.tight_layout()
f.savefig('../figures/fig1B.pdf',bbox_inches='tight')

# Data Log-transformation

In [ ]:
data = load_data()
data = to_msa(data)
data = data_log_transform(data)


In [ ]:
len(np.unique(data['MSA'])),len(np.unique(data['Year']))

In [ ]:
data.shape

In [ ]:
data[data.MSA=='Chicago, IL'].head()

In [ ]:
west_states=['WA','MT','AK','HI','OR','ID','WY','CO','CA','NV','UT','AZ','NM']
midwest_states=['ND','MN','WI','MI','SD','NE','KS','IA','MO','IL','IN','OH']
northeast_states=['ME','VT','NH','MA','RI','NY','CT','PA','MD','NJ']
south_states=['TX','OK','AR','LA','MS','KY','TN','AL','WV','VA','DE','DC','NC','SC','GA','FL']


map_state_regions=dict()
for state in west_states:
    map_state_regions[state]='West'
    
for state in midwest_states:
    map_state_regions[state]='Midwest'

for state in northeast_states:
    map_state_regions[state]='Northeast'
    
for state in south_states:
    map_state_regions[state]= 'South'

data['Regions']=data['State'].map(map_state_regions)
data=data.dropna()

In [ ]:
data.head()

In [ ]:
data.to_csv('../data/log_df.csv')

In [ ]:
exog_var = ['Whites', 'Blacks', 'Minorities', 'Poverty','Unemployment', 'Education','Non-White']
def center_data(data,exog_var):
    data_c=data.copy()
    for column in exog_var:
        data_c[column]=data_c[column]-data_c[column].mean()
    return data_c

data_c = center_data(data,exog_var)

In [ ]:
data_c.to_csv('../data/log_df_centered.csv')

In [ ]:
data_c.head()

# Cross-sectional urban scaling laws (cross-sectional pooling)

In [ ]:
data.head()

In [ ]:
len(np.unique(data['MSA']))

In [ ]:
databyyear=dict()
for year in range(1960,2021):
    databyyear[year]=data[data.Year==year]

In [ ]:
df=databyyear[2010]
df.head()

## Figure 2A: Scaling laws 

In [ ]:
df=databyyear[2010]

df.head()

In [ ]:
urban_metrics=['Whites', 'Blacks', 'Minorities','Poverty','Unemployment','Education', 'Murder']

In [ ]:
from src import stats_functions as sf

In [ ]:
results=[]
for j,metric in enumerate(urban_metrics):
    print(metric)
    y=df[metric].values
    x=df['Population'].values

    param_bounds=([-np.inf,0.1],[np.inf,3])
    p_linear , pcov_linear = optimize.curve_fit(sf.linear, x, y,bounds=param_bounds)

    b,binf,bsup=sf.bootstrapping(x,y,sf.linmodel_slope,100,0.01)
    results.append([metric,round(b,3),round(binf,3),round(bsup,3)])

In [ ]:
df_beta=pd.DataFrame(results,columns=['Metric','Beta','CI2.5','CI97.5'])
df_beta

In [ ]:
colors=palettable.colorbrewer.qualitative.Set1_4.hex_colors 

In [ ]:
dependent_var='Murder'
f=plt.figure(figsize=stdfigsize(nx=3, ny=1.7, scale=0.75))
gs = f.add_gridspec(2, 5)

ax = f.add_subplot(gs[0:2,0:2])
metric='Murder'
y=df[metric].values
x=df['Population'].values
ax.plot(x,y,'o',color='#e31a1c',alpha=0.6,markersize=10)

param_bounds=([-np.inf,0.1],[np.inf,3])
p_linear , pcov_linear = optimize.curve_fit(sf.linear, x, y,bounds=param_bounds)


xr=np.array([4.5,6.5])
yr=sf.linear(np.array([4.5,6.5]),*p_linear)
# Calculate triangle coordinates values
x_mid = np.convolve(xr, [0.5, 0.5], mode='valid')
x_tri = np.vstack((x_mid, x_mid + 0.5))
y_tri = np.interp(x_tri, xr, yr)

ax.plot(x_tri+0.1, np.tile(y_tri[0, :], [2, 1]), '#525252')      # red horizontal line
ax.plot(np.tile(x_tri[1, :], [2, 1])+0.1, y_tri, '#525252')      # red vertical line

b,binf,bsup=sf.bootstrapping(x,y,sf.linmodel_slope,3,0.05)
ax.annotate(r"$\beta=$ {:.2f}".format(b),
        xy=(x_tri[0][0]+0.2,0.5),
        fontsize=30,
        color='k')


r,rinf,rsup=sf.bootstrapping(y,sf.linear(x, *p_linear),r2_score,3,0.05)

ax.annotate(r"$R^2=$ {:.2f}".format(r),
            xy=(0.05, 0.9), 
            xycoords="axes fraction",
            fontsize=30,
            color='k',zorder=-1)

# ax.set_title('Murders',fontsize=30)
ax.set_xlabel('Population, $\log~L$')
ax.set_ylabel('Number of murders, $\log~H$')
# ax.set_ylim(-0.1,3)

xd = np.linspace(min(x), max(x), 100)
ax.plot(xd,sf.linear(xd, *p_linear),"--",color='k',linewidth=5)
ax.annotate('A', xy=(-0.25, 1.0), xycoords="axes fraction",fontsize=40)

ax1 = f.add_subplot(gs[0,2])
ax2 = f.add_subplot(gs[0,3])
ax3 = f.add_subplot(gs[0,4])
ax4 = f.add_subplot(gs[1,2])
ax5 = f.add_subplot(gs[1,3])
ax6 = f.add_subplot(gs[1,4])

ax1.set_ylabel('Urban metrics, $\log~Y_k$')
ax5.set_xlabel('Population, $\log~L$')

axs=[ax1,ax2,ax3,ax4,ax5,ax6]

yp=[5.4, 4., 3.5, 4.1, 3.9, 4.6]
colors=['#878787','#1f78b4','#6a3d9a',
        '#e377c2','#8c564b','#33a02c'] # Murder:'#e31a1c'
for j,metric in enumerate(['Whites', 'Blacks', 'Minorities',
                           'Unemployment','Education','Poverty']):
    ax = axs[j]

    y=df[metric].values
    x=df['Population'].values
    ax.plot(x,y,'o',color=colors[j],alpha=0.3,markersize=10)

    param_bounds=([-np.inf,0.1],[np.inf,3])
    p_linear , pcov_linear = optimize.curve_fit(sf.linear, x, y,bounds=param_bounds)


    xr=np.array([4.5,6.5])
    yr=sf.linear(np.array([4.3,6.3]),*p_linear)
    # Calculate triangle coordinates values
    x_mid = np.convolve(xr, [0.5, 0.5], mode='valid')
    x_tri = np.vstack((x_mid, x_mid + 0.5))
    y_tri = np.interp(x_tri, xr, yr)

    ax.plot(x_tri+0.1, np.tile(y_tri[0, :], [2, 1]), '#525252')      # red horizontal line
    ax.plot(np.tile(x_tri[1, :], [2, 1])+0.1, y_tri, '#525252')      # red vertical line
    
    b,binf,bsup=sf.bootstrapping(x,y,sf.linmodel_slope,3,0.05)
    ax.annotate(r"$\beta=$ {:.2f}".format(b),
                xy=(x_tri[0][0]-0.3,yp[j]-1.55),
                fontsize=30,
                color='k')
    
    
    r,rinf,rsup=sf.bootstrapping(y,sf.linear(x, *p_linear),r2_score,3,0.05)

    ax.annotate(r"$R^2=$ {:.2f}".format(r),
                    xy=(0.05, 0.9), 
                    xycoords="axes fraction",
                    fontsize=30,
                    color='k',zorder=-1)


    xd = np.linspace(min(x), max(x), 100)
    ax.plot(xd,sf.linear(xd, *p_linear),"--",color='k',linewidth=5)
    


    if metric!='Education':
        ax.set_title(ticksnamemap[metric],fontsize=30,pad=30)
    else:
        ax.set_title('Pop. w/o JH diploma',fontsize=30,pad=30)
        
        

f.tight_layout()
f.savefig('../figures/fig2A.pdf',bbox_inches='tight')

## Figure 2B: Scaling exponent evolution (1960-2020)

In [ ]:
urban_metrics=['Whites', 'Blacks', 'Minorities','Poverty','Unemployment','Education', 'Murder','Non-White']
beta = None
try: 
    b=dict()
    for i,metric in enumerate(urban_metrics):
        df=pd.read_csv('../results/cross_sectional/new_beta_{}.csv'.format(metric.lower()),index_col=0)
        b[metric]=df
except:
    n_samples=1000 # use 1000 for paper
    np.random.seed(0)
    beta=[]
    for year in range(1960,2021):
        print(year)
        b=[]
        df=databyyear[year]
        for metric in urban_metrics:
            #try:
            x=df['Population']
            y=df[metric]

            x,y=np.array(x),np.array(y)
            param_bounds=([-np.inf,0.5],[np.inf,3])
            p_linear , pcov_linear = optimize.curve_fit(sf.linear, x, y,bounds=param_bounds)
            ainf,asup, binf,bsup=sf.linmodel_ci(df,x_label='Population',y_label=metric,n_samples=n_samples,alpha=0.05)
            b.append([p_linear[1], binf,bsup])

        beta.append(b)


    


In [ ]:
if beta is not None:
    for i,metric in enumerate(urban_metrics):
        beta_metric=np.transpose([np.transpose(beta)[0][i],np.transpose(beta)[1][i],np.transpose(beta)[2][i]])
        pd.DataFrame(beta_metric,columns=['beta','betainf','betasup']).to_csv('../results/cross_sectional/new_beta_{}.csv'.format(metric.lower()))
    b = {metric: pd.read_csv('../results/cross_sectional/new_beta_{}.csv'.format(metric.lower()), index_col=0)
         for metric in urban_metrics}

In [ ]:
b['Non-White'].head()

In [ ]:
colors=['#878787','#1f78b4','#6a3d9a',#'#dd1c77',
        '#33a02c','#e377c2',
        '#8c564b',
        '#e31a1c']

In [ ]:
ticksnamemap['Non-White']='Non-White'

In [ ]:
min_year=1960

f, axs = plt.subplots(nrows=1, ncols=3, figsize=stdfigsize(nx=3, ny=1.2, scale=0.75),sharey=True)

count=0
# Populations panel
ax = axs.flatten()[0]
for k,metric in enumerate(['Whites', 'Blacks', 'Minorities']):

    beta=b[metric]['beta'].values
    betainf=b[metric]['betainf'].values
    betasup=b[metric]['betasup'].values
    ax.fill_between(range(min_year,2021),betasup,betainf,color=colors[count],alpha=0.2)
    ax.plot(range(min_year,2021),beta,label=ticksnamemap[metric],color=colors[count])
    count=count+1
ax.set_ylim(0.75,1.6)
x_ticks = np.arange(min_year, 2021, 10)
ax.set_xticks(x_ticks)

ax.legend(fontsize=23,labelspacing=0.1,
          frameon=False,bbox_to_anchor=[0.0, -0.05],
          ncol=2,columnspacing=0.1,loc='lower left')
ax.axhline(1,linestyle='dashed',linewidth=3,color='k')
ax.set_ylabel(r'Exponent, $\beta_k$'); 
ax.set_xlabel(r'Year, $t$')
ax.set_title('Demographic')
ax.annotate('B', xy=(-0.25, 1.1), xycoords="axes fraction",fontsize=40)



# POVERTY panel
ax = axs.flatten()[1]
for k,metric in enumerate(['Poverty','Unemployment','Education']):

    beta=b[metric]['beta'].values
    betainf=b[metric]['betainf'].values
    betasup=b[metric]['betasup'].values
    ax.fill_between(range(min_year,2021),betasup,betainf,color=colors[count],alpha=0.2)
    ax.plot(range(min_year,2021),beta,label=ticksnamemap[metric],color=colors[count])
    count=count+1


x_ticks = np.arange(min_year, 2021, 10)
ax.set_xticks(x_ticks)

ax.legend(fontsize=24,labelspacing=0.1,frameon=False,loc='upper right')
ax.axhline(1,linestyle='dashed',linewidth=3,color='k')
ax.set_xlabel(r'Year, $t$')
ax.set_title('Socio-economic')
ax.annotate('C', xy=(-0.1, 1.1), xycoords="axes fraction",fontsize=40)

# CRIME PANEL
ax = axs.flatten()[2]
for k,metric in enumerate(['Murder']):

    beta=b[metric]['beta'].values
    betainf=b[metric]['betainf'].values
    betasup=b[metric]['betasup'].values
    ax.fill_between(range(min_year,2021),betasup,betainf,color=colors[count],alpha=0.3)

    ax.plot(range(min_year,2021),beta,label=ticksnamemap[metric],color=colors[count])
    count=count+1
ax.axhline(1,linestyle='dashed',linewidth=3,color='k')   
x_ticks = np.arange(min_year, 2021, 10)
ax.set_xticks(x_ticks)
ax.set_xlabel(r'Year, $t$')
ax.set_title('Murders')
ax.annotate('D', xy=(-0.1, 1.1), xycoords="axes fraction",fontsize=40)

plt.tight_layout(w_pad=-2)


In [ ]:
min_year=1960

f, axs = plt.subplots(nrows=1, ncols=3, figsize=stdfigsize(nx=3, ny=1.2, scale=0.75),sharey=True)

count=0
# Populations panel
ax = axs.flatten()[0]
for k,metric in enumerate(['Whites', 'Blacks', 'Minorities']):

    beta=b[metric]['beta'].values
    betainf=b[metric]['betainf'].values
    betasup=b[metric]['betasup'].values
    ax.fill_between(range(min_year,2021),betasup,betainf,color=colors[count],alpha=0.2)
    ax.plot(range(min_year,2021),beta,label=ticksnamemap[metric],color=colors[count])
    count=count+1
ax.annotate('White', xy=(0.75,0.15), xycoords="axes fraction",fontsize=30, color = colors[0])
ax.annotate('B&AA', xy=(0.5,0.85), xycoords="axes fraction",fontsize=30, color = colors[1])
ax.annotate('Other', xy=(0.35,0.4), xycoords="axes fraction",fontsize=30, color = colors[2])

ax.set_ylim(0.75,1.6)
x_ticks = np.arange(min_year, 2021, 10)
ax.set_xticks(x_ticks)

ax.axhline(1,linestyle='dashed',linewidth=3,color='k')
ax.set_ylabel(r'Exponent, $\beta_k$'); 
ax.set_xlabel(r'Year, $t$')
ax.set_title('Demographic')
ax.annotate('B', xy=(-0.25, 1.1), xycoords="axes fraction",fontsize=40)





# POVERTY panel
ax = axs.flatten()[1]
for k,metric in enumerate(['Poverty','Unemployment','Education']):

    beta=b[metric]['beta'].values
    betainf=b[metric]['betainf'].values
    betasup=b[metric]['betasup'].values
    ax.fill_between(range(min_year,2021),betasup,betainf,color=colors[count],alpha=0.2)
    ax.plot(range(min_year,2021),beta,label=ticksnamemap[metric],color=colors[count])
    count=count+1

ax.annotate('Poverty ', xy=(0.75,0.11), xycoords="axes fraction",fontsize=30, color = colors[3])
ax.annotate('Unemployed', xy=(0.06,0.4), xycoords="axes fraction",fontsize=30, color = colors[4])
ax.annotate('w/o JH diploma', xy=(0.50,0.50), xycoords="axes fraction",fontsize=30, color = colors[5])

x_ticks = np.arange(min_year, 2021, 10)
ax.set_xticks(x_ticks)

ax.axhline(1,linestyle='dashed',linewidth=3,color='k')
ax.set_xlabel(r'Year, $t$')
ax.set_title('Socio-economic')
ax.annotate('C', xy=(-0.1, 1.1), xycoords="axes fraction",fontsize=40)

# CRIME PANEL
ax = axs.flatten()[2]
for k,metric in enumerate(['Murder']):

    beta=b[metric]['beta'].values
    betainf=b[metric]['betainf'].values
    betasup=b[metric]['betasup'].values
    ax.fill_between(range(min_year,2021),betasup,betainf,color=colors[count],alpha=0.3)

    ax.plot(range(min_year,2021),beta,label=ticksnamemap[metric],color=colors[count])
    count=count+1
ax.axhline(1,linestyle='dashed',linewidth=3,color='k')   
x_ticks = np.arange(min_year, 2021, 10)
ax.set_xticks(x_ticks)
ax.set_xlabel(r'Year, $t$')
ax.set_title('Murders')
ax.annotate('D', xy=(-0.1, 1.1), xycoords="axes fraction",fontsize=40)

plt.tight_layout(w_pad=-2)
plt.savefig('../figures/fig2B.pdf',bbox_inches='tight')


# END:From here on, run the '07_bayesian_..' file on quest 